In [1]:
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast, Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import load_dataset, Dataset
from peft import get_peft_model, LoraConfig, TaskType, PeftModel
import torch

In [2]:
!unzip mbart_lora_es_pt.zip -d mbart_lora_es_pt

Archive:  mbart_lora_es_pt.zip
   creating: mbart_lora_es_pt/mbart_lora_es_pt/
  inflating: mbart_lora_es_pt/__MACOSX/._mbart_lora_es_pt  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/adapter_model.safetensors  
  inflating: mbart_lora_es_pt/__MACOSX/mbart_lora_es_pt/._adapter_model.safetensors  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/tokenizer_config.json  
  inflating: mbart_lora_es_pt/__MACOSX/mbart_lora_es_pt/._tokenizer_config.json  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/special_tokens_map.json  
  inflating: mbart_lora_es_pt/__MACOSX/mbart_lora_es_pt/._special_tokens_map.json  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/sentencepiece.bpe.model  
  inflating: mbart_lora_es_pt/__MACOSX/mbart_lora_es_pt/._sentencepiece.bpe.model  
  inflating: mbart_lora_es_pt/mbart_lora_es_pt/tokenizer.json  
  inflating: mbart_lora_es_pt/__MACOSX/mbart_lora_es_pt/._tokenizer.json  
   creating: mbart_lora_es_pt/mbart_lora_es_pt/checkpoint-3375/
  inflating: mbart_lora_es_pt/

In [2]:
model_name = "facebook/mbart-large-50-many-to-many-mmt"
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
base_model = MBartForConditionalGeneration.from_pretrained(model_name)
model = PeftModel.from_pretrained(base_model, "./mbart_lora_es_pt/mbart_lora_es_pt")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [3]:
model.train()
for name, param in model.named_parameters():
    if "lora" in name:
        param.requires_grad = True

model.print_trainable_parameters()

trainable params: 1,179,648 || all params: 612,059,136 || trainable%: 0.1927


In [4]:
src_lang = "es_XX"
tgt_lang = "pt_XX"
tokenizer.src_lang = src_lang

# Función de traducción
def traducir(texto):
    inputs = tokenizer(texto, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Indicamos que la secuencia de salida debe comenzar en portugués
    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id[tgt_lang],
        max_length=128
    )

    traduccion = tokenizer.decode(generated_tokens[0], skip_special_tokens=True)
    return traduccion

In [5]:
frases = [
    "Hola mundo",
    "¿Cómo estás?",
    "Me gusta aprender cosas nuevas.",
    "Estoy entrenando un modelo de traducción.",
    "La inteligencia artificial es fascinante."
]

for f in frases:
    print(f"ES: {f}")
    print(f"PT: {traducir(f)}\n")

ES: Hola mundo
PT: Olá a todos.

ES: ¿Cómo estás?
PT: Como é que você tem?

ES: Me gusta aprender cosas nuevas.
PT: Eu gosto de aprender coisas novas.

ES: Estoy entrenando un modelo de traducción.
PT: Estou fazendo um modelo de tradução.

ES: La inteligencia artificial es fascinante.
PT: A inteligência artificial é fascinante.



In [7]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 16.2 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12

In [7]:
import huggingface_hub
huggingface_hub.login() # now you will be prompted to enter your token; enter it.

In [8]:
ds_es = load_dataset("openlanguagedata/flores_plus", "spa_Latn", split="dev")
ds_lit = load_dataset("openlanguagedata/flores_plus", "kat_Geor", split="dev")
parallel_lit = [{"translation": {"es": e["text"], "kat": g["text"]}} for e, g in zip(ds_es, ds_lit)]

dataset_lit = Dataset.from_list(parallel_lit).train_test_split(test_size=0.1, seed=42)
train_lit = dataset_lit["train"]
eval_lit = dataset_lit["test"]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

spa_Latn.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

spa_Latn.parquet:   0%|          | 0.00/134k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

...

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

kat_Geor.parquet:   0%|          | 0.00/171k [00:00<?, ?B/s]

kat_Geor.parquet:   0%|          | 0.00/177k [00:00<?, ?B/s]

Generating dev split: 0 examples [00:00, ? examples/s]

Generating devtest split: 0 examples [00:00, ? examples/s]

In [9]:
tokenizer.src_lang = "es_XX"
tokenizer.tgt_lang = "ka_GE"

def tokenize_gl(batch):
    src = [x["es"] for x in batch["translation"]]
    tgt = [x["kat"] for x in batch["translation"]]
    inputs = tokenizer(src, truncation=True, padding="max_length", max_length=128)
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(tgt, truncation=True, padding="max_length", max_length=128)
    inputs["labels"] = labels["input_ids"]
    return inputs

train_tokenized_lit = train_lit.map(tokenize_gl, batched=True)
eval_tokenized_lit = eval_lit.map(tokenize_gl, batched=True)

Map:   0%|          | 0/897 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/tokenization_utils_base.py:3951: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [10]:
training_args_lit = Seq2SeqTrainingArguments(
    output_dir="./mbart_lora_es_ka",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=5e-4,
    num_train_epochs=15,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs_ka",
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    save_total_limit=1,
    report_to="none",
    label_names=["labels"]
)

trainer_lit = Seq2SeqTrainer(
    model=model,
    args=training_args_lit,
    train_dataset=train_tokenized_lit,
    eval_dataset=eval_tokenized_lit,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
)

In [11]:
trainer_lit.train()
model.save_pretrained("./mbart_lora_es_ka")
tokenizer.save_pretrained("./mbart_lora_es_ka")

Epoch,Training Loss,Validation Loss
1,No log,8.203176
2,No log,8.174928
3,8.117200,8.176414
4,8.117200,8.176278
5,7.928500,8.188328
6,7.928500,8.196384
7,7.838800,8.202806
8,7.838800,8.215504
9,7.733300,8.228054
10,7.733300,8.232557


('./mbart_lora_es_ka/tokenizer_config.json',
 './mbart_lora_es_ka/special_tokens_map.json',
 './mbart_lora_es_ka/sentencepiece.bpe.model',
 './mbart_lora_es_ka/added_tokens.json',
 './mbart_lora_es_ka/tokenizer.json')

In [12]:
pip install evaluate sacrebleu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 13.0 MB/s eta 0:00:00


In [13]:
from evaluate import load

In [14]:
def test_translation_batch(sentences_es, references_gl, model_path="./mbart_lora_es_ka"):
    try:
        # Cargar modelo base + adaptadores LoRA
        base_model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50-many-to-many-mmt")
        model = PeftModel.from_pretrained(base_model, model_path)
        model.eval()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model.to(device)

        # Cargar tokenizer
        tokenizer = MBart50TokenizerFast.from_pretrained(model_path)
        inputs = tokenizer(sentences_es, return_tensors="pt", padding=True, truncation=True).to(device)

        # Generar traducciones
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=128,
                num_beams=4,
                early_stopping=True,
                do_sample=False
            )

        translations = [tokenizer.decode(out, skip_special_tokens=True) for out in outputs]

        # Mostrar resultados
        for src, pred, ref in zip(sentences_es, translations, references_gl):
            print(f"ES: {src}")
            print(f"KA (pred): {pred}")
            print(f"KA (ref):  {ref}")
            print("-" * 60)

        # Calcular métricas BLEU y chrF en lote
        print("\n📊 Métricas globales:")
        bleu = load("bleu")
        chrf = load("chrf")

        bleu_score = bleu.compute(predictions=translations, references=[[r] for r in references_gl])
        chrf_score = chrf.compute(predictions=translations, references=references_gl)

        print(f"BLEU: {bleu_score['bleu']:.4f}")
        print(f"chrF: {chrf_score['score']:.2f}")

    except Exception as e:
        print(f"❌ Error en traducción: {e}")

In [15]:
ds_es = load_dataset("openlanguagedata/flores_plus", "spa_Latn", split="devtest[:50]")
ds_lit = load_dataset("openlanguagedata/flores_plus", "kat_Geor", split="devtest[:50]")

# Tomar una muestra de 50 ejemplos para visualización
sample_es = ds_es["text"][:50]
sample_lit = ds_lit["text"][:50]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/220 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/214 [00:00<?, ?it/s]

In [16]:
test_translation_batch(sample_es, sample_lit)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


ES: «Actualmente, tenemos ratones de cuatro meses de edad que antes solían ser diabéticos y que ya no lo son», agregó.
KA (pred): ამჟამად ჩვენ გვაქვს 4 თვის შვილები, რომლებიც თავიდან დიაბეტი იყო და ახლა დიაბეტი აღარ არის," დასძინა.
KA (ref):  „ჩვენ ახლა გვყავს 4 თვის ასაკის თაგვები, რომლებსაც დიაბეტი ჰქონდათ და ახლა აღარ აქვთ,“ — დასძინა მან.
------------------------------------------------------------
ES: La investigación todavía se ubica en su etapa inicial, conforme indicara el Dr. Ehud Ur, docente en la carrera de medicina de la Universidad de Dalhousie, en Halifax, Nueva Escocia, y director del departamento clínico y científico de la Asociación Canadiense de Diabetes.
KA (pred): კვლევის ჯერ კიდევ პირველ ეტაპზეა, როგორც მიუთითებდა ჰალაფაქსში, ნიუ-სკოლანდიის უნივერსიტეტის დელუუსის სამედიცინო პროფესორმა და დიაბეტის კანადის ასოციაციის კლინიკური და მეცნიერული დეპარტამენტის დირექტორმა ეჰიდ ორმა.
KA (ref):  ჰალიფაქსის, ახალი შოტლანდიის დალჰუსის უნივერსიტეტის სამედიცინო განყოფილების პროფე

BLEU: 0.0546
chrF: 37.81


In [17]:
import shutil
from google.colab import files

# Comprimir la carpeta (reemplaza 'nombre_de_tu_carpeta')
shutil.make_archive('mbart_lora_es_ka', 'zip', 'mbart_lora_es_ka')

# Descargar el archivo ZIP
files.download('mbart_lora_es_ka.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>